In [2]:
import pandas as pd
Jvirrs = pd.read_csv("../data/raw/fire_archive_J1V-C2_816387.csv")

Jnrtvirrs = pd.read_csv("../data/raw/fire_nrt_J1V-C2_816387.csv")

Svirrs = pd.read_csv("../data/raw/fire_archive_SV-C2_816388.csv")

Snrtvirrs = pd.read_csv("../data/raw/fire_nrt_SV-C2_816388.csv")

Jvirrs["origen"] = "archive_J1V-C2"

Jnrtvirrs["origen"] = "nrt_J1V-C2_816387"

Svirrs["origen" ] = "archive_SV-C2"

Snrtvirrs["origen"] = "nrt_SV-C2_816388"

union_csv = pd.concat([Jvirrs, Jnrtvirrs, Svirrs, Snrtvirrs],axis=0,ignore_index=True )

union_csv["confidence"].value_counts()



confidence
n    22576
l     1341
h      671
Name: count, dtype: int64

In [3]:
datos_confiables = union_csv[union_csv["confidence"] != "l"]
print(datos_confiables.shape)

(23247, 16)


In [ ]:
duplicados = datos_confiables.duplicated()
print(duplicados.sum())

0


In [5]:
columnas_finales = ["latitude", "longitude", "acq_date", "acq_time", "satellite",
                     "bright_t31", "brightness", "frp", "daynight", "type", "origen"]
datos_limpios = datos_confiables[columnas_finales]
datos_limpios.info()

<class 'pandas.DataFrame'>
Index: 23247 entries, 0 to 24587
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   latitude    23247 non-null  float64
 1   longitude   23247 non-null  float64
 2   acq_date    23247 non-null  str    
 3   acq_time    23247 non-null  int64  
 4   satellite   23247 non-null  str    
 5   bright_t31  23247 non-null  float64
 6   brightness  23247 non-null  float64
 7   frp         23247 non-null  float64
 8   daynight    23247 non-null  str    
 9   type        22668 non-null  float64
 10  origen      23247 non-null  str    
dtypes: float64(6), int64(1), str(4)
memory usage: 2.1 MB


In [6]:
datos_limpios.to_csv("../data/proccesed/incendios_limpios.csv", index= False)

In [7]:
data_ec = pd.read_csv("../data/proccesed/incendios_limpios.csv")


In [8]:
zona_dmq = datos_limpios[(datos_limpios["latitude"]<= -0.05) & (datos_limpios["latitude"]>= -0.5) & (datos_limpios["longitude"]>= -78.70) & (datos_limpios["longitude"]<= -78.25)]
zona_junin = datos_limpios[(datos_limpios['latitude']<= -0.80) & (datos_limpios["latitude"]>= -1.00)&(datos_limpios["longitude"]<= -80.05) & (datos_limpios["longitude"]>= -80.30)]
print(zona_dmq.shape)
print(zona_junin.shape)

(827, 11)
(750, 11)


In [9]:
zona_dmq["zona"] = "DMQ"
zona_junin["zona"] = "Junin"
print(zona_junin.shape)


(750, 12)


In [10]:
zona = pd.concat([zona_dmq, zona_junin], axis=0, ignore_index= True)
print(zona.shape)


(1577, 12)


In [11]:
print(zona.dtypes)
zona.head()


latitude      float64
longitude     float64
acq_date          str
acq_time        int64
satellite         str
bright_t31    float64
brightness    float64
frp           float64
daynight          str
type          float64
origen            str
zona              str
dtype: object


,latitude,longitude,acq_date,acq_time,satellite,bright_t31,brightness,frp,daynight,type,origen,zona
0,-0.36008,-78.44814,2021-01-02,1809,N20,290.81,329.74,4.63,D,0.0,archive_J1V-C2,DMQ
1,-0.34528,-78.26051,2021-02-07,1834,N20,301.38,345.07,12.63,D,0.0,archive_J1V-C2,DMQ
2,-0.34197,-78.26104,2021-02-07,1834,N20,301.08,350.03,12.63,D,0.0,archive_J1V-C2,DMQ
3,-0.33318,-78.26607,2021-02-08,715,N20,275.92,300.93,1.97,N,0.0,archive_J1V-C2,DMQ
4,-0.33387,-78.26170,2021-02-08,715,N20,277.54,319.78,1.97,N,0.0,archive_J1V-C2,DMQ


In [12]:
zona.to_csv("../data/proccesed/zona_incendios.csv", index=False)

In [13]:
import requests
respuesta= requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=-0.275&longitude=-78.475&start_date=2021-01-01&end_date=2026-10-04&daily=temperature_2m_max,temperature_2m_min,precipitation_sum")
datos_meteo =respuesta.json()
#print(datos_meteo)
print(datos_meteo.keys())
print(datos_meteo["daily"].keys())

import pandas as pd
clima_dmq = pd.DataFrame(datos_meteo["daily"])
print(clima_dmq.shape)
print(clima_dmq.head())

import requests 
respuesta_2 = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=-0.90&longitude=-80.175&start_date=2021-01-01&end_date=2026-10-04&daily=temperature_2m_max,temperature_2m_min,precipitation_sum")
datos_meteo_2 = respuesta_2.json()

import pandas as pd
clima_junin = pd.DataFrame(datos_meteo_2["daily"])
print(clima_junin.head())

clima_junin["zona"] = "Junin"
clima_dmq["zona"] = "DMQ"
climas_ec_total = pd.concat([clima_junin,clima_dmq],axis=0, ignore_index=True)
print(climas_ec_total.shape)

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum'])
(2103, 4)
         time  temperature_2m_max  temperature_2m_min  precipitation_sum
0  2021-01-01                19.3                 7.2                0.5
1  2021-01-02                18.4                 9.5                3.8
2  2021-01-03                17.4                10.7                2.1
3  2021-01-04                16.2                10.7                7.7
4  2021-01-05                17.0                11.2                9.5
         time  temperature_2m_max  temperature_2m_min  precipitation_sum
0  2021-01-01                28.0                21.1                3.3
1  2021-01-02                28.1                21.7               19.2
2  2021-01-03                28.6                21.8                6.3
3  2021-01-04        

In [14]:
print(zona["acq_date"].iloc[0])
print(climas_ec_total["time"].iloc[0])

2021-01-02
2021-01-01


In [15]:
#merge_zonas = pd.merge(zona,climas_ec_total, left_on=["acq_date", "zona"], right_on=["time", "zona"], how="inner")
#print(merge_zonas.head())
#print(merge_zonas.shape)

In [16]:
zona["acq_date"] = pd.to_datetime(zona["acq_date"],format= "%Y-%m-%d")
climas_ec_total["time"] = pd.to_datetime(climas_ec_total["time"],format= "%Y-%m-%d")
print(zona["acq_date"].dtype)
print(climas_ec_total["time"].dtype)

datetime64[us]
datetime64[us]


In [17]:
climas_ec_total["semana"] = climas_ec_total["time"].dt.to_period("W")
climas_ec_total.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,zona,semana
0,2021-01-01,28.0,21.1,3.3,Junin,2020-12-28/2021-01-03
1,2021-01-02,28.1,21.7,19.2,Junin,2020-12-28/2021-01-03
2,2021-01-03,28.6,21.8,6.3,Junin,2020-12-28/2021-01-03
3,2021-01-04,27.0,21.2,0.4,Junin,2021-01-04/2021-01-10
4,2021-01-05,28.4,21.5,1.5,Junin,2021-01-04/2021-01-10


In [18]:
climas_zonas_semanas = climas_ec_total.groupby(["zona", "semana"]).agg({
    "temperature_2m_max": "max",
    "temperature_2m_min": "mean",
    "precipitation_sum": "sum"
})
print(climas_zonas_semanas.shape)
climas_zonas_semanas.head()
climas_zonas_semanas["temperature_prom_max"] = climas_ec_total.groupby(["zona", "semana"])["temperature_2m_max"].mean()
climas_zonas_semanas.head()
print(climas_zonas_semanas.shape)

(602, 3)
(602, 4)


In [19]:
zona["semana"] = zona["acq_date"].dt.to_period("W")
zona.head()

,latitude,longitude,acq_date,acq_time,satellite,bright_t31,brightness,frp,daynight,type,origen,zona,semana
0,-0.36008,-78.44814,2021-01-02,1809,N20,290.81,329.74,4.63,D,0.0,archive_J1V-C2,DMQ,2020-12-28/2021-01-03
1,-0.34528,-78.26051,2021-02-07,1834,N20,301.38,345.07,12.63,D,0.0,archive_J1V-C2,DMQ,2021-02-01/2021-02-07
2,-0.34197,-78.26104,2021-02-07,1834,N20,301.08,350.03,12.63,D,0.0,archive_J1V-C2,DMQ,2021-02-01/2021-02-07
3,-0.33318,-78.26607,2021-02-08,715,N20,275.92,300.93,1.97,N,0.0,archive_J1V-C2,DMQ,2021-02-08/2021-02-14
4,-0.33387,-78.26170,2021-02-08,715,N20,277.54,319.78,1.97,N,0.0,archive_J1V-C2,DMQ,2021-02-08/2021-02-14


In [20]:
incendios_por_semana = zona.groupby(["zona","semana"])["latitude"].count()
print(incendios_por_semana.shape)
incendios_por_semana.head()



(240,)


zona  semana               
DMQ   2020-12-28/2021-01-03    1
      2021-01-11/2021-01-17    1
      2021-02-01/2021-02-07    2
      2021-02-08/2021-02-14    2
      2021-03-01/2021-03-07    3
Name: latitude, dtype: int64

In [21]:
incendios_por_semana.sum()
incendios_semanas = incendios_por_semana.reset_index(name="n_incendios")
climas_semanas =climas_zonas_semanas.reset_index()
incendios_semanas.head()


,zona,semana,n_incendios
0,DMQ,2020-12-28/2021-01-03,1
1,DMQ,2021-01-11/2021-01-17,1
2,DMQ,2021-02-01/2021-02-07,2
3,DMQ,2021-02-08/2021-02-14,2
4,DMQ,2021-03-01/2021-03-07,3


In [22]:
climas_semanas.head()

,zona,semana,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_prom_max
0,DMQ,2020-12-28/2021-01-03,19.3,9.133333,6.4,18.366667
1,DMQ,2021-01-04/2021-01-10,20.5,9.757143,39.3,17.585714
2,DMQ,2021-01-11/2021-01-17,20.4,8.128571,7.2,18.971429
3,DMQ,2021-01-18/2021-01-24,20.2,8.442857,15.1,19.028571
4,DMQ,2021-01-25/2021-01-31,21.9,8.757143,6.4,19.885714


In [23]:
dataset_ec = pd.merge(left=climas_semanas, right=incendios_semanas, how="left", on=['semana',"zona"])
dataset_ec.shape
dataset_ec.head()

,zona,semana,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_prom_max,n_incendios
0,DMQ,2020-12-28/2021-01-03,19.3,9.133333,6.4,18.366667,1.0
1,DMQ,2021-01-04/2021-01-10,20.5,9.757143,39.3,17.585714,NaN
2,DMQ,2021-01-11/2021-01-17,20.4,8.128571,7.2,18.971429,1.0
3,DMQ,2021-01-18/2021-01-24,20.2,8.442857,15.1,19.028571,NaN
4,DMQ,2021-01-25/2021-01-31,21.9,8.757143,6.4,19.885714,NaN


In [24]:
dataset_ec["n_incendios"].isna().sum()

np.int64(362)

In [25]:
incendios_por_semana.shape[0]
dataset_ec["n_incendios"].sum()

np.float64(1577.0)

In [26]:
dataset_ec["n_incendios"]= dataset_ec["n_incendios"].fillna(0)
dataset_ec.head()

,zona,semana,temperature_2m_max,temperature_2m_min,precipitation_sum,temperature_prom_max,n_incendios
0,DMQ,2020-12-28/2021-01-03,19.3,9.133333,6.4,18.366667,1.0
1,DMQ,2021-01-04/2021-01-10,20.5,9.757143,39.3,17.585714,0.0
2,DMQ,2021-01-11/2021-01-17,20.4,8.128571,7.2,18.971429,1.0
3,DMQ,2021-01-18/2021-01-24,20.2,8.442857,15.1,19.028571,0.0
4,DMQ,2021-01-25/2021-01-31,21.9,8.757143,6.4,19.885714,0.0


In [27]:
dataset_ec["hubo_incendio"] = (dataset_ec["n_incendios"]>0).astype(int)
dataset_ec["hubo_incendio"].value_counts()

hubo_incendio
0    362
1    240
Name: count, dtype: int64

In [28]:
dataset_ec = dataset_ec.rename(columns={"temperature_2m_max":"temp_max_semana","temperature_2m_min": "temp_min_semana","precipitation_sum":"precip_semana", "temperature_prom_max":"temp_prom_max_semana"})
dataset_ec.to_csv("../data/proccesed/dataset_semanal.csv", index=False)
dataset_ec.columns
dataset_ec["n_incendios"].isna().sum()


np.int64(0)

In [29]:
X = dataset_ec[["temp_max_semana", "temp_min_semana", 'precip_semana', "temp_prom_max_semana"]]
y = dataset_ec["hubo_incendio"]

X.shape
y.shape

(602,)

In [30]:

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape
X_test.shape

(121, 4)

In [31]:
y_test.value_counts()

hubo_incendio
0    73
1    48
Name: count, dtype: int64

In [32]:
from sklearn.linear_model import LogisticRegression
modelo = LogisticRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
print(y_pred[:10])
X_train.shape

[1 0 1 0 1 0 1 0 0 0]


(481, 4)

In [33]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.74      0.81      0.77        73
           1       0.66      0.56      0.61        48

    accuracy                           0.71       121
   macro avg       0.70      0.69      0.69       121
weighted avg       0.71      0.71      0.71       121



In [34]:
media = X_train.mean()
desv = X_train.std()
X_train_n = (X_train-media)/desv
X_test_n = (X_test-media)/desv
X_train_n.describe()

,temp_max_semana,temp_min_semana,precip_semana,temp_prom_max_semana
count,4.810000e+02,4.810000e+02,4.810000e+02,4.810000e+02
mean,9.011041e-16,-5.908879e-17,-2.215830e-17,-8.124709e-16
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.958232e+00,-1.584364e+00,-8.062710e-01,-1.764844e+00
25%,-9.833724e-01,-9.737062e-01,-6.814414e-01,-9.965708e-01
50%,-8.512292e-03,-4.492864e-01,-4.185025e-01,2.881399e-02
75%,9.013571e-01,1.021420e+00,3.331311e-01,9.287040e-01
max,1.919544e+00,1.338402e+00,4.649048e+00,1.727586e+00


In [35]:
X_train_n.shape

(481, 4)

In [36]:
import torch 
X_train_t = torch.tensor(X_train_n.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1,1)
X_test_t = torch.tensor(X_test_n.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1,1)
print(X_train_t.shape, y_train_t.shape, X_test_t.shape, y_test_t.shape)

torch.Size([481, 4]) torch.Size([481, 1]) torch.Size([121, 4]) torch.Size([121, 1])


In [37]:
from torch import nn
from torch.nn import ReLU
torch.manual_seed(42)
red = nn.Sequential(
    nn.Linear(4,8),
    ReLU(),
    nn.Linear(8,1) 
)
print(red)


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)


In [38]:
loss_fn = nn.BCEWithLogitsLoss() 
optim = torch.optim.SGD(red.parameters(),lr= 0.1)

In [39]:
for epoch in range(500):
    salida = red(X_train_t)
    loss = loss_fn(salida, y_train_t)
    optim.zero_grad()
    loss.backward()
    optim.step()
    if epoch % 100 == 0 :
        print(epoch, loss.item())


0 0.8080115914344788
100 0.6028517484664917
200 0.5490014553070068
300 0.5267025232315063
400 0.5169109106063843


In [40]:
with torch.no_grad():
    logits = red(X_test_t)
    probs = torch.sigmoid(logits)
    y_pred_red = (probs > 0.4).int()
print(classification_report(y_test,y_pred_red))

              precision    recall  f1-score   support

           0       0.79      0.71      0.75        73
           1       0.62      0.71      0.66        48

    accuracy                           0.71       121
   macro avg       0.70      0.71      0.70       121
weighted avg       0.72      0.71      0.71       121



In [41]:
paquete = {
    "pesos": red.state_dict(),
    "media": media.to_list(),
    "desv": desv.tolist(),
    "umbral": 0.4,
    "columnas": list(X.columns),
}
torch.save(paquete, "../modelo_riesgo.pt")

In [42]:
cargado = torch.load("../modelo_riesgo.pt", weights_only=False)
print(cargado.keys())
print(cargado["umbral"], cargado["columnas"])

dict_keys(['pesos', 'media', 'desv', 'umbral', 'columnas'])
0.4 ['temp_max_semana', 'temp_min_semana', 'precip_semana', 'temp_prom_max_semana']
